[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/doxav/astromodel_proving/blob/main/analysis/07_assumption_sensitivity.ipynb)

# Step 07 - Assumption sensitivity: gating, proxy, and compartment split

This notebook runs the local Step 07 pipeline without Google Drive dependencies and writes auditable outputs under `outputs/assumption_sensitivity/`.

**Scope:** reviewer-facing assumption checks use all accepted cells by default with one best accepted candidate per cell. This keeps the assumption screen tied to the full cell-level validation scope without arbitrary row caps.

**Claim scope:** Step 07 audits whether Step 04-06 conclusions are robust to modeling assumptions. It does not authorize final biological degeneracy wording because parameter plausibility and synthesis checks remain pending.

In [1]:
from pathlib import Path
import os
import pandas as pd
import matplotlib.pyplot as plt

PROJECT_ROOT = Path(os.environ.get("ASTROMODEL_PROJECT_ROOT", ".")).resolve()
PROJECT_ROOT

PosixPath('/home/xav/code/astromodel_proving')

In [2]:
from src.step07_assumption_sensitivity import Step07Config, run_step07_assumption_sensitivity

config = Step07Config(
    max_candidates=None,
    candidate_policy="best_per_cell",
    time_points=40,
    write_outputs=True,
)
result = run_step07_assumption_sensitivity(PROJECT_ROOT, config)
result["analysis_summary"]

/home/xav/code/astromodel_proving/src/astro_model.py:311: ODEintWarning: Excess work done on this call (perhaps wrong Dfun type). Run with full_output = 1 to get quantitative information.
  states = odeint(_rhs_for_odeint, z0_arr, t_arr, args=(paramdict,))


/home/xav/code/astromodel_proving/src/astro_model.py:311: ODEintWarning: Excess work done on this call (perhaps wrong Dfun type). Run with full_output = 1 to get quantitative information.
  states = odeint(_rhs_for_odeint, z0_arr, t_arr, args=(paramdict,))


/home/xav/code/astromodel_proving/src/astro_model.py:311: ODEintWarning: Excess work done on this call (perhaps wrong Dfun type). Run with full_output = 1 to get quantitative information.
  states = odeint(_rhs_for_odeint, z0_arr, t_arr, args=(paramdict,))


{'step_name': 'Step 07 — Assumption sensitivity: gating, proxy, and compartment split',
 'config': {'max_candidates': None,
  'candidate_policy': 'best_per_cell',
  'time_points': 40,
  't_final_ms': 50000.0,
  'gating_families': ('sigmoid',
   'tanh',
   'hill',
   'soft_threshold',
   'hard_threshold',
   'double_sigmoid'),
  'currents_na': (100,),
  'proxy_corr_min': 0.5,
  'proxy_rmse_max': 1.25,
  'gating_divergence_rmse_max_mV': 8.0,
  'heldout_pass_min': 0.3,
  'write_outputs': True},
 'n_candidates': 30,
 'n_gating_rows': 180,
 'n_proxy_rows': 30,
 'n_compartment_rows': 30,
 'gating_families': ['sigmoid',
  'tanh',
  'hill',
  'soft_threshold',
  'hard_threshold',
  'double_sigmoid'],
 'claim_scope': 'Step 07 can mark assumptions robust or model-dependent, but final biological degeneracy claims remain disallowed until later plausibility/statistical checks pass.',
 'elapsed_seconds': 5.314636347000487}

## Accepted ensemble inventory

The Step 07 input contract preserves cell identity, region, condition, candidate ID, and mechanism labels from upstream steps.

In [3]:
gating = result["gating_family_comparison"]
inventory = gating[["file_id", "region", "condition", "candidate_id", "mechanism_cluster", "dominant_mechanism"]].drop_duplicates()
inventory.head(12)

,file_id,region,condition,candidate_id,mechanism_cluster,dominant_mechanism
0,1_DH_1_CONTROL,DH,CONTROL,1_DH_1_CONTROL__hybrid_optuna_087,M2,Mixed
6,1_DH_2_CONTROL,DH,CONTROL,1_DH_2_CONTROL__hybrid_optuna_098,M2,Mixed
12,1_VH_1_CONTROL,VH,CONTROL,1_VH_1_CONTROL__hybrid_optuna_004,M1,Mixed
18,1_VH_2_CONTROL,VH,CONTROL,1_VH_2_CONTROL__hybrid_optuna_001,M1,Mixed
24,2_DH_1_CONTROL,DH,CONTROL,2_DH_1_CONTROL__hybrid_optuna_096,M2,Mixed
30,3_DH_1_CONTROL,DH,CONTROL,3_DH_1_CONTROL__hybrid_optuna_097,M2,Mixed
36,3_DH_2_CONTROL,DH,CONTROL,3_DH_2_CONTROL__hybrid_optuna_098,M2,Mixed
42,DH_1_CONTROL,DH,CONTROL,DH_1_CONTROL__hybrid_optuna_099,M2,Mixed
48,DH_1_MFA,DH,MFA,DH_1_MFA__hybrid_optuna_057,M3,Mixed
54,DH_1_MFA_BA,DH,MFA_BA,DH_1_MFA_BA__hybrid_optuna_094,M1,Mixed


## Gating-family comparison

All configured gating families are scored using the same candidate/current/time-grid contract.

In [4]:
gating[["gating_family", "region", "condition", "candidate_id", "current_na", "simulation_status", "trace_rmse_vs_sigmoid_mV", "family_supported_under_same_contract", "mechanism_claim_stable"]].head(18)

,gating_family,region,condition,candidate_id,current_na,simulation_status,trace_rmse_vs_sigmoid_mV,family_supported_under_same_contract,mechanism_claim_stable
0,sigmoid,DH,CONTROL,1_DH_1_CONTROL__hybrid_optuna_087,100,ok,0.000000,True,True
1,tanh,DH,CONTROL,1_DH_1_CONTROL__hybrid_optuna_087,100,ok,0.000308,True,True
2,hill,DH,CONTROL,1_DH_1_CONTROL__hybrid_optuna_087,100,ok,0.004809,True,True
3,soft_threshold,DH,CONTROL,1_DH_1_CONTROL__hybrid_optuna_087,100,ok,0.004809,True,True
4,hard_threshold,DH,CONTROL,1_DH_1_CONTROL__hybrid_optuna_087,100,ok,0.004809,True,True
5,double_sigmoid,DH,CONTROL,1_DH_1_CONTROL__hybrid_optuna_087,100,ok,0.002258,True,True
6,sigmoid,DH,CONTROL,1_DH_2_CONTROL__hybrid_optuna_098,100,ok,0.000000,True,True
7,tanh,DH,CONTROL,1_DH_2_CONTROL__hybrid_optuna_098,100,ok,0.000098,True,True
8,hill,DH,CONTROL,1_DH_2_CONTROL__hybrid_optuna_098,100,ok,0.001802,True,True
9,soft_threshold,DH,CONTROL,1_DH_2_CONTROL__hybrid_optuna_098,100,ok,0.001802,True,True


In [5]:
fig, ax = plt.subplots(figsize=(8, 4))
plot_df = gating[gating["simulation_status"].eq("ok")].copy()
if not plot_df.empty:
    plot_df.boxplot(column="trace_rmse_vs_sigmoid_mV", by="gating_family", ax=ax, rot=30)
    ax.set_ylabel("RMSE versus sigmoid baseline (mV)")
    ax.set_title("Gating-family divergence under identical contract")
    fig.suptitle("")
else:
    ax.text(0.5, 0.5, "No successful gating simulations", ha="center")
plt.tight_layout()

## Model-comparison summary

In [6]:
model = result["model_comparison"]
model

,model_family,assumption_axis,n_candidate_current_rows,n_successful_simulations,mean_trace_rmse_vs_sigmoid_mV,median_step04_trace_rmse_mV,mean_heldout_pass_fraction,same_split_same_loss_contract,mechanism_stability_fraction,claim_scope
0,double_sigmoid,gating_form,30,30,0.931686,6.597484,0.454236,step07_same_candidates_currents_timegrid_loss_v1,1.000000,robust_across_configured_gating_families
1,hard_threshold,gating_form,30,30,1.957287,6.597484,0.454236,step07_same_candidates_currents_timegrid_loss_v1,0.966667,model_dependent_or_insufficient_evidence
2,hill,gating_form,30,30,1.699642,6.597484,0.454236,step07_same_candidates_currents_timegrid_loss_v1,0.966667,model_dependent_or_insufficient_evidence
3,sigmoid,gating_form,30,30,0.000000,6.597484,0.454236,step07_same_candidates_currents_timegrid_loss_v1,1.000000,robust_across_configured_gating_families
4,soft_threshold,gating_form,30,30,1.957287,6.597484,0.454236,step07_same_candidates_currents_timegrid_loss_v1,0.966667,model_dependent_or_insufficient_evidence
5,tanh,gating_form,30,30,0.012469,6.597484,0.454236,step07_same_candidates_currents_timegrid_loss_v1,1.000000,robust_across_configured_gating_families


## Intracellular K proxy validity

The local intracellular proxy `ΔK_a,t` is compared with simulated extracellular `K_o` using correlation, scaled RMSE, and lag metrics. Rows that fail criteria explicitly require an ECS variant or additional data.

In [7]:
proxy = result["proxy_validity_by_ensemble"]
proxy[["file_id", "region", "condition", "candidate_id", "current_na", "pearson_r", "spearman_r", "scaled_rmse", "best_lag_samples", "proxy_validity_status", "explicit_ecs_variant_required"]].head(12)

,file_id,region,condition,candidate_id,current_na,pearson_r,spearman_r,scaled_rmse,best_lag_samples,proxy_validity_status,explicit_ecs_variant_required
0,1_DH_1_CONTROL,DH,CONTROL,1_DH_1_CONTROL__hybrid_optuna_087,100,0.498688,0.772420,0.866782,-4,proxy_limited,True
1,1_DH_2_CONTROL,DH,CONTROL,1_DH_2_CONTROL__hybrid_optuna_098,100,0.500064,0.772420,0.865988,-4,proxy_supported,False
2,1_VH_1_CONTROL,VH,CONTROL,1_VH_1_CONTROL__hybrid_optuna_004,100,-0.455372,-0.449906,0.890301,8,proxy_limited,True
3,1_VH_2_CONTROL,VH,CONTROL,1_VH_2_CONTROL__hybrid_optuna_001,100,-0.425680,-0.389118,0.904874,8,proxy_limited,True
4,2_DH_1_CONTROL,DH,CONTROL,2_DH_1_CONTROL__hybrid_optuna_096,100,0.500064,0.772420,0.865988,-4,proxy_supported,False
5,3_DH_1_CONTROL,DH,CONTROL,3_DH_1_CONTROL__hybrid_optuna_097,100,0.500064,0.772420,0.865988,-4,proxy_supported,False
6,3_DH_2_CONTROL,DH,CONTROL,3_DH_2_CONTROL__hybrid_optuna_098,100,0.500064,0.772420,0.865988,-4,proxy_supported,False
7,DH_1_CONTROL,DH,CONTROL,DH_1_CONTROL__hybrid_optuna_099,100,0.500064,0.772420,0.865988,-4,proxy_supported,False
8,DH_1_MFA,DH,MFA,DH_1_MFA__hybrid_optuna_057,100,-0.359434,-0.417448,0.933171,8,proxy_limited,True
9,DH_1_MFA_BA,DH,MFA_BA,DH_1_MFA_BA__hybrid_optuna_094,100,0.271301,-0.154034,0.962495,-7,proxy_limited,True


In [8]:
fig, ax = plt.subplots(figsize=(6, 4))
if not proxy.empty:
    ax.scatter(proxy["pearson_r"], proxy["scaled_rmse"], c=proxy["explicit_ecs_variant_required"].astype(int), cmap="coolwarm", s=80)
    ax.axvline(config.proxy_corr_min, color="black", linestyle="--", linewidth=1, label="corr threshold")
    ax.axhline(config.proxy_rmse_max, color="gray", linestyle=":", linewidth=1, label="RMSE threshold")
    ax.set_xlabel("Pearson r: ΔK_a,t vs K_o")
    ax.set_ylabel("Scaled RMSE after linear rescaling")
    ax.legend()
else:
    ax.text(0.5, 0.5, "No proxy rows", ha="center")
plt.tight_layout()

## Compartment-split sensitivity

The two-state local proxy is compared with a one-state aggregate proxy. This is a sensitivity score rather than a replacement model fit.

In [9]:
split = result["compartment_split_sensitivity"]
split[["file_id", "region", "condition", "candidate_id", "current_na", "two_state_proxy_abs_corr", "one_state_proxy_abs_corr", "corr_delta_one_minus_two", "split_sensitivity_status", "mechanism_structure_persists"]].head(12)

,file_id,region,condition,candidate_id,current_na,two_state_proxy_abs_corr,one_state_proxy_abs_corr,corr_delta_one_minus_two,split_sensitivity_status,mechanism_structure_persists
0,1_DH_1_CONTROL,DH,CONTROL,1_DH_1_CONTROL__hybrid_optuna_087,100,0.498688,0.499050,3.623805e-04,split_robust,True
1,1_DH_2_CONTROL,DH,CONTROL,1_DH_2_CONTROL__hybrid_optuna_098,100,0.500064,0.500200,1.356787e-04,split_robust,True
2,1_VH_1_CONTROL,VH,CONTROL,1_VH_1_CONTROL__hybrid_optuna_004,100,0.455372,0.455372,0.000000e+00,split_robust,True
3,1_VH_2_CONTROL,VH,CONTROL,1_VH_2_CONTROL__hybrid_optuna_001,100,0.425680,0.238110,-1.875704e-01,split_robust,True
4,2_DH_1_CONTROL,DH,CONTROL,2_DH_1_CONTROL__hybrid_optuna_096,100,0.500064,0.500200,1.356787e-04,split_robust,True
5,3_DH_1_CONTROL,DH,CONTROL,3_DH_1_CONTROL__hybrid_optuna_097,100,0.500064,0.500200,1.356787e-04,split_robust,True
6,3_DH_2_CONTROL,DH,CONTROL,3_DH_2_CONTROL__hybrid_optuna_098,100,0.500064,0.500200,1.356787e-04,split_robust,True
7,DH_1_CONTROL,DH,CONTROL,DH_1_CONTROL__hybrid_optuna_099,100,0.500064,0.500200,1.356787e-04,split_robust,True
8,DH_1_MFA,DH,MFA,DH_1_MFA__hybrid_optuna_057,100,0.359434,0.359434,-1.012890e-11,split_robust,True
9,DH_1_MFA_BA,DH,MFA_BA,DH_1_MFA_BA__hybrid_optuna_094,100,0.271301,0.489995,2.186937e-01,split_robust,True


In [10]:
fig, ax = plt.subplots(figsize=(6, 4))
if not split.empty:
    ax.scatter(split["two_state_proxy_abs_corr"], split["one_state_proxy_abs_corr"], s=80)
    ax.plot([0, 1], [0, 1], color="black", linestyle="--", linewidth=1)
    ax.set_xlabel("Two-state |corr(ΔK_a,t, K_o)|")
    ax.set_ylabel("One-state |corr(ΔK_a,total, K_o)|")
    ax.set_xlim(0, 1.05)
    ax.set_ylim(0, 1.05)
else:
    ax.text(0.5, 0.5, "No compartment rows", ha="center")
plt.tight_layout()

## Conservative claim scope

Final biological degeneracy language remains disallowed after Step 07. The table below separates robust, model-dependent, and unresolved assumption axes.

In [11]:
claims = result["claim_scope_table"]
claims

,assumption_axis,status,final_degeneracy_claim_allowed_after_step07
0,gating_form,model_dependent_or_insufficient_evidence,False
1,intracellular_K_as_ECS_proxy,explicit_ecs_variant_or_extra_data_needed,False
2,local_syncytial_compartment_split,split_robust,False


In [12]:
output_dir = PROJECT_ROOT / "outputs" / "assumption_sensitivity"
print("Wrote:")
for path in sorted(output_dir.glob("*")):
    print("-", path.relative_to(PROJECT_ROOT))

Wrote:
- outputs/assumption_sensitivity/analysis_summary.json
- outputs/assumption_sensitivity/claim_scope_table.csv
- outputs/assumption_sensitivity/compartment_split_sensitivity.csv
- outputs/assumption_sensitivity/gating_family_comparison.csv
- outputs/assumption_sensitivity/model_comparison.csv
- outputs/assumption_sensitivity/proxy_validity_by_ensemble.csv


In [13]:
assert result["analysis_summary"]["n_candidates"] >= 1
assert not result["claim_scope_table"]["final_degeneracy_claim_allowed_after_step07"].astype(bool).any()
print("Step 07 full-cell assumption sensitivity notebook completed across the full cell-level target scope.")

Step 07 full-cell assumption sensitivity notebook completed across the full cell-level target scope.


## Post-execution scientific status

Executed status for reviewer response: Step 07 evaluated 30 candidates across gating, proxy, and compartment-split assumption screens. The local/syncytial compartment split is `split_robust`, but gating remains `model_dependent_or_insufficient_evidence` and the intracellular-K proxy still requires explicit ECS-variant or extra-data support. This directly answers R3 with an objective limitation rather than an overclaim; final degeneracy wording remains disallowed after Step 07.